# Reproduce `translate`

Generated from `<inference page: translate x 1 instance(s)>`. The notebook replays inference + evaluation against the exact config snapshot that produced the original run, so a re-execution should produce a comparable `model_patch` (LLM determinism caveats notwithstanding).

- **Instances:** 1
- **Config id:** `translate`


## 1. Setup

The notebook's `kernelspec.name = "evomas"` (see metadata at the bottom of the file) tells Jupyter / VSCode to auto-pick the interpreter `install.ps1` / `install.sh` registered for `~/.evomas-venv`. As a safety net the first cell also prepends the venv's site-packages to `sys.path` — so even if the kernel falls back to a generic Python 3 (different machine, no `install.ps1` run), the evomas imports still resolve. Adjust `OLLAMA_BASE_URL` if your Ollama daemon isn't on the default host; `SWEBENCH_API_KEY` is only required by the remote-eval cell at the bottom.

### Picking the kernel in VSCode

If VSCode opens the notebook outside the EvoMas workspace (e.g. straight from `~/Downloads`), it won't auto-resolve the kernelspec and asks you to **Select Kernel**. Two-tier picker:

- **"Python Environments…"** lists raw Python interpreters discovered by the Python extension (system Python, conda envs, `.venv`/`venv` folders inside workspaces). `~/.evomas-venv` is outside the conventional discovery paths, so it does NOT show up here.
- **"Jupyter Kernel…"** lists registered Jupyter kernelspecs (`%APPDATA%\jupyter\kernels\*` on Windows, `~/.local/share/jupyter/kernels/*` on Linux/mac). This is where the EvoMas one lives — pick **"Python 3 (EvoMas)"** here. VSCode remembers the choice per-notebook so you only have to do it once.

If the entry doesn't appear there: `Ctrl+Shift+P` → **"Developer: Reload Window"** so the Jupyter extension re-scans kernelspecs, or run `jupyter kernelspec list` to confirm `evomas` is registered (if not, re-run `install.ps1` / `install.sh`).

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Defensive sys.path prepend: if the running kernel isn't the
# evomas-venv one (e.g. user opened the notebook on a fresh
# clone without running install.ps1, or VSCode picked a generic
# Python 3), surface the venv's site-packages so `import
# evomas...` still resolves. Skipped when the active sys
# already points at the venv.
_venv = Path.home() / '.evomas-venv'
if _venv.is_dir() and str(_venv) not in sys.executable:
    for _sp in (_venv / 'Lib' / 'site-packages',
                _venv / 'lib' / 'site-packages'):
        if _sp.is_dir() and str(_sp) not in sys.path:
            sys.path.insert(0, str(_sp))

import evomas.paths  # noqa: F401  # triggers load_dotenv(evomas/.env)
from evomas.core.workflow.runner import run as run_evomas
from evomas.utils.instances import fetch_swebench_instances

# Route Python `logging` records to BOTH the notebook output
# AND a per-run text log so `experiments/generate_report.py`
# can mine handoffs / tool calls / per-LLM-call tokens from
# the same lines the API matrix path writes. `force=True`
# overrides any prior basicConfig (e.g. from a stale kernel)
# so the format actually takes effect.
import logging
RUN_OUTPUT_DIR = Path('notebook-translate').resolve()
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_OUTPUT_DIR / 'inference.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    force=True,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ],
)
print(f'Mirroring inference logs to {LOG_FILE}')


Mirroring inference logs to C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\inference.log


### Environment variables

All run-time configuration the notebook needs lives in this cell — edit values here rather than chasing them through the code. Each variable falls back to the value the EvoMas .env file / shell already has set (via `os.environ.setdefault`), so the cell is safe to re-run.

- **`OLLAMA_BASE_URL`** — where the Ollama daemon serves. Local default works when `ollama serve` runs on this machine; swap to e.g. `http://192.168.1.50:11434` for a remote host.
- **`SWEBENCH_API_KEY`** — required by the remote-eval cell (section 5) when running `--remote` against sb-cli. Local Docker harness runs don't need it.
- **`EVOMAS_INSTANCES`** — override the SWE-bench instance cache location. The next cell already searches sensible defaults; only set this if your cache is somewhere non-standard.
- **`GOOGLE_API_KEY` / `OPENAI_API_KEY`** — only needed if the inlined config picks a Gemini / OpenAI model instead of Ollama.


In [2]:
# Edit these inline OR export them in your shell before
# launching Jupyter. `setdefault` means values already in the
# environment (e.g. loaded from evomas/.env) win.
os.environ['OLLAMA_BASE_URL'] = 'http://192.168.1.50:11434'
# os.environ.setdefault('SWEBENCH_API_KEY', 'swb_...')
# os.environ.setdefault('EVOMAS_INSTANCES', '/path/to/swebench_instances.jsonl')
# os.environ.setdefault('GOOGLE_API_KEY',   '...')
# os.environ.setdefault('OPENAI_API_KEY',   '...')

# Echo the effective values (mask secrets) so you can verify the cell ran.
for _k in ('OLLAMA_BASE_URL', 'SWEBENCH_API_KEY', 'EVOMAS_INSTANCES',
           'GOOGLE_API_KEY', 'OPENAI_API_KEY'):
    _v = os.environ.get(_k, '')
    if not _v:
        print(f'  {_k:<18} <unset>')
    elif _k.endswith('_API_KEY'):
        print(f'  {_k:<18} {_v[:6]}***({len(_v)} chars)')
    else:
        print(f'  {_k:<18} {_v}')


  OLLAMA_BASE_URL    http://192.168.1.50:11434
  SWEBENCH_API_KEY   swb_QM***(56 chars)
  EVOMAS_INSTANCES   <unset>
  GOOGLE_API_KEY     AIzaSy***(39 chars)
  OPENAI_API_KEY     <unset>


## 2. Inlined config

Exact resolved config the original run used. Tweak hyperparameters here if you want to experiment with variations.

The cell below the config dict renders a mermaid diagram of the topology so you can see the agent-graph shape at a glance. The diagram is regenerated from `CONFIG['edges']` + `CONFIG['agents']` every time the cell runs, so edits to the dict above are reflected immediately.

In [3]:
CONFIG = {   'id': 'translate',
    'description': 'File-translation pipeline: locator finds the target file, translator writes '
                   'the translated body to disk, reviewer flags untranslated chunks or formatting '
                   "drift. The runner's `generate_diff_impl(workspace)` picks up the workspace "
                   "mutation as the final patch — no finalizer needed since there's nothing to "
                   'assemble. All three agents use the role-less `Base agent` type — prompts and '
                   'tools come from this JSON, no Python subclassing. Language pair is read from '
                   '`instance.problem_statement` (e.g. `Translate from English to Spanish`).',
    'entry': 'locator',
    'end': ['reviewer'],
    'edges': [{'from': 'locator', 'to': 'translator'}, {'from': 'translator', 'to': 'reviewer'}],
    'agents': {   'locator': {   'class': 'Base agent',
                                 'model': 'ollama/qwen3.5:4b',
                                 'think': False,
                                 'num_ctx': 4096,
                                 'temperature': 0.2,
                                 'top_p': 0.9,
                                 'num_predict': 256,
                                 'stop': ['</files>'],
                                 'max_iters': 4,
                                 'prompts': {   'system': 'You locate the single source file the '
                                                          'user wants translated. The workspace '
                                                          'contains exactly the files that need to '
                                                          'be translated; ignore anything whose '
                                                          'name ends in `.gold` (those are '
                                                          'reference translations and MUST NOT be '
                                                          'touched). Use `list_files` and '
                                                          '`read_file` to inspect, then emit the '
                                                          'absolute path of the file to translate, '
                                                          'wrapped in <files>…</files>. Emit '
                                                          'nothing else after the closing tag.',
                                                'user': 'Workspace: {workspace_path}\n'
                                                        '\n'
                                                        'Task:\n'
                                                        '{issue_text}\n'
                                                        '\n'
                                                        'Find the file to translate. Output:\n'
                                                        '<files>\n'
                                                        '<absolute path>\n'
                                                        '</files>'},
                                 'tools': [{'name': 'list_files'}, {'name': 'read_file'}]},
                  'translator': {   'class': 'Base agent',
                                    'model': 'ollama/qwen3.5:4b',
                                    'think': True,
                                    'num_ctx': 16384,
                                    'temperature': 0,
                                    'top_p': 0.9,
                                    'num_predict': 4096,
                                    'stop': [],
                                    'max_iters': 8,
                                    'prompts': {   'system': 'You translate prose files between '
                                                             'human languages. Read the file path '
                                                             'passed from the locator (inside '
                                                             '<files>). Read the file content with '
                                                             '`read_file`. Translate ALL prose '
                                                             'between the source and target '
                                                             'languages named in the task. Hard '
                                                             'rules:\n'
                                                             '- Preserve markdown structure '
                                                             'verbatim (headings, lists, code '
                                                             'fences, links, tables).\n'
                                                             '- DO NOT translate code blocks, '
                                                             'inline code, URLs, identifiers, or '
                                                             'filenames.\n'
                                                             '- DO NOT add commentary, prefaces, '
                                                             "or 'here is the translation' "
                                                             'framing.\n'
                                                             '- Call `write_file(path, content)` '
                                                             'ONCE per file. `path` is just the '
                                                             'filename (e.g. `intro.md`) -- the '
                                                             'tool sandboxes writes to the active '
                                                             'workspace automatically. `content` '
                                                             'is the FULL translated body.\n'
                                                             'After the write_file call succeeds, '
                                                             'reply with a single confirmation '
                                                             'line of the form `Translated '
                                                             '<relative-path> from <src> to '
                                                             '<tgt>.` -- nothing else.',
                                                   'user': 'Files to translate: {locator}\n'
                                                           '\n'
                                                           'Task (source/target languages '
                                                           'described here):\n'
                                                           '{issue_text}'},
                                    'tools': [{'name': 'read_file'}, {'name': 'write_file'}]},
                  'reviewer': {   'class': 'Base agent',
                                  'model': 'ollama/qwen3.5:4b',
                                  'think': True,
                                  'num_ctx': 8192,
                                  'temperature': 0,
                                  'top_p': 0.9,
                                  'num_predict': 1024,
                                  'stop': ['</review>'],
                                  'max_iters': 4,
                                  'prompts': {   'system': 'Review the translation the translator '
                                                           'wrote. Read the modified file with '
                                                           "`read_file` (path is in the locator's "
                                                           '<files> block). Check three things '
                                                           'ONLY:\n'
                                                           '1. The whole document is now in the '
                                                           'TARGET language (no leftover '
                                                           'source-language paragraphs).\n'
                                                           '2. Markdown / formatting is intact (no '
                                                           'broken code fences, no dropped '
                                                           'headings).\n'
                                                           '3. Code blocks, URLs, and identifiers '
                                                           'were left untranslated.\n'
                                                           'If the translation is acceptable, emit '
                                                           '`<review>APPROVED</review>`. If not, '
                                                           'list the specific issues inside '
                                                           '<review>…</review>. Do not rewrite the '
                                                           'file — your job is only to flag.',
                                                 'user': 'Files reviewed: {locator}\n'
                                                         '\n'
                                                         'Task:\n'
                                                         '{issue_text}\n'
                                                         '\n'
                                                         'Workspace: {workspace_path}\n'
                                                         '\n'
                                                         'Produce your review.'},
                                  'tools': [{'name': 'read_file'}]}}}

In [4]:
from IPython.display import Markdown, display

def _topology_mermaid(cfg):
    """Render the topology as a Mermaid flowchart.

    Mirrors what the topology page's cytoscape canvas shows:
    virtual START/END boundary nodes, one node per agent with
    its class as a second-line label, edges directed left-to-
    right. Renders inline in Jupyter Lab + VSCode Jupyter; if
    the cell falls back to plain text the source stays readable.
    """
    lines = ['graph LR']
    lines.append('    START((START))')
    lines.append('    END((END))')
    for name, block in (cfg.get('agents') or {}).items():
        cls = (block or {}).get('class', '') or ''
        label = f'{name}<br/><i>{cls}</i>' if cls else name
        # Backticks would break the mermaid parser; strip them
        # defensively. Class names never contain them today,
        # this is just future-proofing.
        label = label.replace('`', '')
        lines.append(f'    {name}["{label}"]')
    entry = cfg.get('entry') or ''
    if entry:
        lines.append(f'    START --> {entry}')
    for e in (cfg.get('edges') or []):
        if isinstance(e, dict) and e.get('from') and e.get('to'):
            lines.append(f'    {e["from"]} --> {e["to"]}')
    end_field = cfg.get('end')
    ends = (
        [end_field] if isinstance(end_field, str) and end_field
        else list(end_field or [])
    )
    # Only emit `→ END` for nodes with no outgoing edges (the
    # same wiring rule `graph_builder.py` uses). Hub-in-end
    # nodes with outgoing edges don't get the static edge.
    out_sources = {e.get('from') for e in (cfg.get('edges') or [])
                   if isinstance(e, dict)}
    for n in ends:
        if n and n not in out_sources:
            lines.append(f'    {n} --> END')
    return '\n'.join(lines)

display(Markdown('```mermaid\n' + _topology_mermaid(CONFIG) + '\n```'))


```mermaid
graph LR
    START((START))
    END((END))
    locator["locator<br/><i>Base agent</i>"]
    translator["translator<br/><i>Base agent</i>"]
    reviewer["reviewer<br/><i>Base agent</i>"]
    START --> locator
    locator --> translator
    translator --> reviewer
    reviewer --> END
```

## 3. Instances

Self-contained: the notebook regenerates its own instances from zero each run. SWE-bench rows get pulled fresh from HuggingFace (cached under `~/.cache/huggingface`); custom rows are reconstructed from the minimal inputs the user added via the Inference page's `+ Custom` modal.

In [5]:
INSTANCE_IDS = ['translate-intro-en-es']


In [6]:
# Pull plan for SWE-bench rows: `{(subset, split): [ids]}`.
# At runtime the cell below calls `fetch_swebench_instances`
# per group and filters down to just these IDs.
SWEBENCH_GROUPS = {}


In [7]:
# Custom-instance inputs (no upstream — added locally via the
# Inference page's `+ Custom` modal). Notebook reconstructs the
# row dict from these fields; nothing else is needed.
CUSTOM_ROWS = [   {   'instance_id': 'translate-intro-en-es',
        'repo': 'EvoMas/translate-demo-intro-en',
        'base_commit': '66da274aa4b243043c5e338765f3b4ad0dac5e8f',
        'problem_statement': 'Translate from English to Spanish. Translate the prose of every '
                             'markdown file in the workspace while preserving headings, lists, '
                             'links, inline code and code fences verbatim. Do NOT modify any file '
                             'whose name ends in .gold — those are reference translations.',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'}]


In [8]:
# Materialise SWE-bench + custom rows into one JSONL the
# runner consumes. Re-uses `RUN_OUTPUT_DIR` from the setup
# cell so inference.log + prediction JSONL share one folder.
output_dir = RUN_OUTPUT_DIR
output_path = output_dir / 'prediction-translate.jsonl'
INSTANCES_PATH = output_dir / 'instances.jsonl'

selected = []
for (subset, split), ids in SWEBENCH_GROUPS.items():
    print(f'Fetching {len(ids)} {subset}/{split} row(s) from HuggingFace…')
    selected.extend(fetch_swebench_instances(subset, split, instance_ids=ids))
selected.extend(CUSTOM_ROWS)

with INSTANCES_PATH.open('w', encoding='utf-8') as _fh:
    for _row in selected:
        _fh.write(json.dumps(_row, ensure_ascii=False) + '\n')
print(f'Wrote {len(selected)} instance row(s) -> {INSTANCES_PATH}')

_have = {i['instance_id'] for i in selected}
missing = [iid for iid in INSTANCE_IDS if iid not in _have]
if missing:
    print('Missing rows (id not found in HF or in CUSTOM_ROWS):', missing)
print(f'Ready to run {len(selected)} instance(s).')


Wrote 1 instance row(s) -> C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\instances.jsonl
Ready to run 1 instance(s).


## 4. Inference

Re-runs the EvoMas workflow for each instance with the inlined config. All notebook-produced artefacts (prediction JSONL, evaluation reports, custom-instance sidecar) land under one per-run folder at `notebook-translate/` so they stay grouped together and don't mix with UI/CLI runs in the repo's `results/` tree.

In [9]:
from evomas.exceptions.errors import OllamaMemoryError

# `output_dir` + `output_path` were created in the instances cell above.
predictions = []
with open(output_path, 'w', encoding='utf-8') as out:
    for inst in selected:
        iid = inst['instance_id']
        print(f'--- {iid} ---')
        try:
            patch = run_evomas(inst, config=CONFIG)
        except OllamaMemoryError as exc:
            print(f'Ollama OOM; aborting: {exc}')
            break
        except Exception as exc:
            print(f'run failed on {iid}: {exc}')
            patch = ''
        rec = {
            'instance_id': iid,
            'model_patch': patch,
            'model_name_or_path': 'evomas-notebook',
        }
        predictions.append(rec)
        out.write(json.dumps(rec) + '\n')
print(f'Wrote {len(predictions)} prediction(s) to {output_path}.')


2026-06-06 11:32:27,453 [WARNING] weave.trace.op: Warning: Traces will not be logged. Call weave.init to log your traces to a project.
 (subsequent messages of this type will be suppressed)


2026-06-06 11:32:27,453 [INFO] evomas.core.workflow.runner: === running translate-intro-en-es with inline config (id=translate) ===


--- translate-intro-en-es ---


2026-06-06 11:32:27,634 [INFO] evomas.utils.workspace: reusing workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\translate-intro-en-es (HEAD=66da274aa4b243043c5e338765f3b4ad0dac5e8f)


2026-06-06 11:32:27,816 [INFO] evomas.core.workflow.runner: graph runtime: 3 agents x 2 max revisits => recursion_limit=6


...

2026-06-06 11:34:15,515 [INFO] evomas.models.langchain_ollama_model: [reviewer|resp ] <review>APPROVED


2026-06-06 11:34:15,515 [INFO] evomas.models.langchain_ollama_model: [reviewer] tokens in=2173 out=330 total=2503


2026-06-06 11:34:15,516 [INFO] evomas.agents.reviewer: [reviewer] no tool calls — stopping loop


2026-06-06 11:34:15,539 [INFO] evomas.core.workflow.runner: === translate-intro-en-es done: 2238-char patch | tokens in=9494 out=2255 total=11749 ===


Wrote 1 prediction(s) to C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\prediction-translate.jsonl.


## 5. Evaluation

Runs `scripts/evaluation/translate_eval.py` — the evaluator chosen at notebook-generation time (the `--evaluator` flag on `evomas notebook`). Forwards through the unified evaluator CLI contract: `--predictions / --instances / --report-dir / --run-id / --model`. Output lands at `<report-dir>/{model}.{run-id}.json` plus per-instance folders under `<report-dir>/logs/run_evaluation/<run-id>/<model>/<instance>/`.

In [10]:
# Evaluator baked at notebook-gen time (--evaluator on `evomas notebook`).
EVALUATOR_STEM = 'translate_eval'
EVALUATOR_NEEDS_WSL = False

first = selected[0] if selected else None
SUBSET = (first or {}).get('subset', 'lite')
SPLIT  = (first or {}).get('split',  'dev')

# All eval artifacts land under `output_dir` (alongside
# instances.jsonl + prediction-*.jsonl).
eval_report_dir = output_dir

import platform
from evomas.paths import BASE_DIR as _BASE_DIR
_script = _BASE_DIR / 'scripts' / 'evaluation' / f'{EVALUATOR_STEM}.py'
if EVALUATOR_NEEDS_WSL and platform.system() == 'Windows':
    from evomas.utils.paths import to_wsl
    cmd = [
        'wsl', '--', 'python3', to_wsl(str(_script)),
        '--predictions', to_wsl(str(output_path)),
        '--instances',   to_wsl(str(INSTANCES_PATH)),
        '--report-dir',  to_wsl(str(eval_report_dir)),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
else:
    cmd = [
        sys.executable, str(_script),
        '--predictions', str(output_path),
        '--instances',   str(INSTANCES_PATH),
        '--report-dir',  str(eval_report_dir),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
print(f'Evaluating via {EVALUATOR_STEM}.py')
print('+ ' + ' '.join(cmd))

eval_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding='utf-8', errors='replace',
)
assert eval_proc.stdout is not None
for line in eval_proc.stdout:
    print(line, end='')
eval_proc.wait()
print(f'\n[evaluation finished with exit code {eval_proc.returncode}]')

# Surface the per-instance artifacts the evaluator wrote.
logs_root = eval_report_dir / 'logs' / 'run_evaluation'
if logs_root.is_dir():
    print('\nPer-instance artifacts:')
    for inst_dir in sorted(logs_root.rglob('*/')):
        if (inst_dir / 'report.json').is_file():
            print(f'  {inst_dir}')
for summary in sorted(eval_report_dir.glob('*.json')):
    print(f'Summary: {summary}')


Evaluating via translate_eval.py
+ C:\Users\XF\.evomas-venv\Scripts\python.exe C:\Users\XF\Desktop\TFG\EvoMas\scripts\evaluation\translate_eval.py --predictions C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\prediction-translate.jsonl --instances C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\instances.jsonl --report-dir C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate --run-id notebook-custom-custom --model evomas-notebook


  -- translate-intro-en-es --
     RESOLVED  bleu=56.55

Resolved 1/1 instances (threshold corpus-BLEU=50.0).
Summary -> C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\evomas-notebook.notebook-custom-custom.json

[evaluation finished with exit code 0]

Per-instance artifacts:
  C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\logs\run_evaluation\notebook-custom-custom\evomas-notebook\translate-intro-en-es
Summary: C:\Users\XF\Desktop\TFG\EvoMas\examples\translate_demo\notebook-translate\evomas-notebook.notebook-custom-custom.json
